# Базовый прогон модели SIR

**Авторы:** А. В. Королькова, PhD, Кулябов Д. С., DSc

**Принадлежность:** Российский университет дружбы народов

## Назначение скрипта

Данный скрипт выполняет один базовый эксперимент с фиксированными параметрами
для модели SIR, реализованной с использованием сетей Петри.

### Что делает скрипт

1. Выполняет два типа симуляции:
   - **Детерминированную** (решение ОДУ) — даёт плавную усреднённую динамику
   - **Стохастическую** (алгоритм Гиллеспи) — учитывает случайные флуктуации

2. Сохраняет результаты в CSV‑файлы

3. Строит и сохраняет графики S(t), I(t), R(t) для обоих типов симуляции

## Параметры модели

| Параметр | Значение | Описание |
|----------|----------|----------|
| β | 0.3 | Коэффициент заражения |
| γ | 0.1 | Коэффициент выздоровления |
| tmax | 100.0 | Время симуляции |
| S₀ | 990 | Начальное число восприимчивых |
| I₀ | 10 | Начальное число инфицированных |
| R₀ | 0 | Начальное число выздоровевших |

**Зерно для ГСЧ:** `Random.seed!(123)` — обеспечивает воспроизводимость стохастического прогона.

## Выходные данные

| Файл | Описание |
|------|----------|
| `data/sir_det.csv` | Детерминированная симуляция (time, S, I, R) |
| `data/sir_stoch.csv` | Стохастическая симуляция (time, S, I, R) |
| `plots/sir_det_dynamics.png` | График динамики (ОДУ) — рис. 6.1 |
| `plots/sir_stoch_dynamics.png` | График динамики (Гиллеспи) — рис. 6.2 |

## Интерпретация результатов

- **Детерминированный график** показывает классический пик эпидемии: рост I,
  максимум, затем спад до нуля; R растёт и выходит на плато, S падает.

- **Стохастический график** может иметь шумы и немного отличаться по времени пика
  и амплитуде — это демонстрирует влияние случайности.

- **Сравнение двух типов симуляции** показывает, что при большом начальном числе
  восприимчивых (990) стохастическая траектория близка к детерминированной,
  но при малых числах различия были бы значительны.

## Инициализация проекта DrWatson

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение генератора случайных чисел

In [ ]:
using Random

## Загрузка модуля SIRPetri

Подключаем модуль с реализацией модели SIR на сетях Петри.
Файл `src/SIRPetri.jl` содержит функции:
- `build_sir_network` — создание сети
- `simulate_deterministic` — детерминированная симуляция
- `simulate_stochastic` — стохастическая симуляция
- `plot_sir` — визуализация результатов

In [ ]:
include(srcdir("SIRPetri.jl"))
using .SIRPetri

## Подключение утилит для работы с данными и графикой

In [ ]:
using DataFrames, CSV, Plots

## Задание параметров модели

In [ ]:
β = 0.3
γ = 0.1
tmax = 100.0

## Создание сети Петри

Функция `build_sir_network` создаёт размеченную сеть с:
- Состояниями: S (Susceptible), I (Infectious), R (Recovered)
- Переходами: infection (S + I → I + I), recovery (I → R)
- Начальной маркировкой: S₀ = 990, I₀ = 10, R₀ = 0

In [ ]:
net, u0, states = build_sir_network(β, γ)

## Детерминированная симуляция

Решает систему обыкновенных дифференциальных уравнений:

```
dS/dt = -β·S·I
dI/dt =  β·S·I - γ·I
dR/dt =  γ·I
```

**Метод решения:** Tsit5 (метод Рунге-Кутты 5-го порядка)
**Шаг сохранения:** saveat = 0.5

In [ ]:
df_det = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.5, rates = [β, γ])

### Сохранение результатов детерминированной симуляции

In [ ]:
CSV.write(datadir("sir_det.csv"), df_det)

## Стохастическая симуляция

Использует прямой алгоритм Гиллеспи (Gillespie SSA):

1. Вычисление пропускных способностей:
   - `a_inf = β·S·I` — заражение
   - `a_rec = γ·I` — выздоровление

2. Время до следующего события: `dt = -ln(r) / (a_inf + a_rec)`

3. Случайный выбор перехода пропорционально весу

**Зерно для воспроизводимости:** 123

In [ ]:
Random.seed!(123)
df_stoch = simulate_stochastic(net, u0, (0.0, tmax), rates = [β, γ])

### Сохранение результатов стохастической симуляции

In [ ]:
CSV.write(datadir("sir_stoch.csv"), df_stoch)

## Визуализация и сохранение графиков

### Рисунок 6.1: Детерминистический график динамики SIR

На графике отображаются три кривые:
- **S(t)** — восприимчивые (синяя линия)
- **I(t)** — инфицированные (красная линия)
- **R(t)** — выздоровевшие (зелёная линия)

Характерные особенности:
- Пик эпидемии (максимум I(t))
- Плато R(t) — итоговое число переболевших
- Монотонное убывание S(t)

In [ ]:
p_det = plot_sir(df_det)
savefig(plotsdir("sir_det_dynamics.png"))

### Рисунок 6.2: Стохастический график динамики SIR

В отличие от детерминированного графика, стохастическая траектория:
- Имеет шумы и флуктуации
- Может немного отличаться по времени пика и амплитуде
- Демонстрирует влияние случайности на динамику эпидемии

In [ ]:
p_stoch = plot_sir(df_stoch)
savefig(plotsdir("sir_stoch_dynamics.png"))

## Завершение работы

In [ ]:
println("Базовый прогон завершён. Результаты в data/ и plots/")